# Kiểm tra crawl Nhà thuốc Long Châu đang chạy

Notebook này kiểm tra `status.duckdb` ở chế độ chỉ đọc và hiển thị shard mới nhất trên Drive. Nó không ghi vào database. Nếu crawler đang giữ khóa ghi, truy vấn status sẽ báo khóa; khi đó snapshot status chưa thể đọc từ runtime khác cho tới lúc crawler commit/nhả khóa. Các status đọc được chỉ gồm những thay đổi đã commit.

In [ ]:
!pip -q install duckdb
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from IPython.display import clear_output, display
import duckdb, time
import pandas as pd

DB_PATH = Path('/content/drive/MyDrive/r2ai_data/crawl/status.duckdb')
HTML_DIR = Path('/content/drive/MyDrive/r2ai_data/crawl/html')
METRICS_PATH = Path('/content/drive/MyDrive/r2ai_data/reports/crawl_metrics_latest.json')
print('DB:', DB_PATH, '| exists:', DB_PATH.exists())
print('Metrics snapshot:', METRICS_PATH, '| exists:', METRICS_PATH.exists())
print('Shard folder:', HTML_DIR, '| exists:', HTML_DIR.exists())

In [ ]:
# Refresh mỗi 15 giây trong 30 phút; dừng cell bằng nút Stop bất cứ lúc nào.
REFRESH_SECONDS = 15
N_REFRESHES = 120

for _ in range(N_REFRESHES):
    clear_output(wait=True)
    if METRICS_PATH.exists():
        try:
            metrics = json.loads(METRICS_PATH.read_text(encoding='utf-8'))
            print('Snapshot cập nhật lúc:', metrics.get('updated_at', 'không rõ'))
            batch_total = metrics.get('active_batch_total', metrics.get('queued', 0))
            print(f"Batch: {metrics.get('processed', 0):,}/{batch_total:,}")
            active_counts = metrics.get('active_batch_status_counts')
            if active_counts is not None:
                print('Status của đúng batch đang crawl (đã commit):')
                display(pd.DataFrame(active_counts))
            else:
                print('Snapshot cũ chưa có status riêng cho batch đang crawl.')
        except Exception as exc:
            print('Chưa đọc được metrics snapshot:', repr(exc))
    else:
        print('Chưa có metrics snapshot:', METRICS_PATH)

    if not DB_PATH.exists():
        print('Không thấy status.duckdb tại:', DB_PATH)
    else:
        db = None
        try:
            db = duckdb.connect(str(DB_PATH), read_only=True)
            result = db.execute("""
                SELECT status, http_code, COUNT(*) AS so_url,
                       COUNT(*) FILTER (WHERE shard IS NOT NULL) AS co_shard
                FROM crawl_status
                WHERE lower(url) LIKE '%nhathuoclongchau.com.vn%'
                GROUP BY status, http_code
                ORDER BY status, http_code
            """).df()
            print('Status toàn bộ Nhà thuốc Long Châu trong DuckDB (đã commit):')
            display(result)
        except Exception as exc:
            print('Chưa đọc được DuckDB:', repr(exc))
            print('Có thể crawler đang giữ khóa ghi; sẽ thử lại ở chu kỳ kế tiếp.')
        finally:
            if db is not None:
                db.close()

    if HTML_DIR.exists():
        shards = sorted(HTML_DIR.glob('shard_*.jsonl.zst'), key=lambda p: p.stat().st_mtime)
        temps = list(HTML_DIR.glob('*.tmp'))
        print('Shard hoàn tất:', len(shards), '| file tạm:', len(temps))
        if shards:
            last = shards[-1]
            print('Shard mới nhất:', last.name, '| bytes:', last.stat().st_size,
                  '| modified:', time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(last.stat().st_mtime)))
    time.sleep(REFRESH_SECONDS)
